# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and exports
trades for a training-selected wallet universe.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [1]:
# TO recalculate: 
# remove token_df file
# remove enhanced trades
# remove processed trades
# remove processed markets

# rm /Users/vobornij/projects/polymarket/data/markets_processed/*.parquet
# rm /Users/vobornij/projects/polymarket/data/trades_polygon_enriched/enriched_*
# rm /Users/vobornij/projects/polymarket/data/polygon_trades_processed/*.parquet

In [2]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [3]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 4, 15)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

The token lookup table (`token_id → condition_id, outcome, token_winner, final_price`) is
built once via `build_token_lookup_df()` and persisted to
`data/markets_processed/token_lookup.parquet`. Subsequent runs load it with
`load_token_lookup_df()` instead of re-parsing all market JSON.

In [4]:
# # Load only the market files whose month falls within [START_DATE, END_DATE].
# import datetime as _dt
# def _file_in_range(p, start, end):
#     """Return True if YYYY-MM.parquet filename falls within the date range."""
#     try:
#         y, m = (int(x) for x in p.stem.split("-"))
#         file_date = _dt.date(y, m, 1)
#         return start.replace(day=1) <= file_date <= end.replace(day=1)
#     except Exception:
#         return False

# market_files = [
#     p for p in sorted(MARKETS_DIR.glob("*.parquet"))
#     if _file_in_range(p, START_DATE, END_DATE)
# ]
# print(f"Found {len(market_files)} market file(s)")

# all_market_rows = pd.concat(
#     [pd.read_parquet(f) for f in market_files],
#     ignore_index=True,
# )
# # deduplicate by condition_id (keep first occurrence)
# all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
# print(f"Unique markets (all):  {len(all_market_rows):,}")

# # ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# # Parse end_date_iso as a date; keep only markets whose resolution date falls
# # within [START_DATE, END_DATE] (inclusive).
# end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
# in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
# all_market_rows = all_market_rows[in_range].reset_index(drop=True)
# print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
# all_market_rows.head(2)

In [5]:
from polymarket_analysis.data.data_catalogue import (
    load_markets_processed,
    build_token_lookup_df,
    load_token_lookup_df,
)
token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

token_df = token_df[
    ~(token_df['primary_tag'].isin(['Sports', 'Crypto']))
    & token_df['token_winner'].notna()
    ]
len(token_df)

Processing 74 market file(s)…
Saved 2,457,346 token rows → /Users/vobornij/projects/polymarket/data/markets_processed/token_lookup.parquet
Token lookup rows: 2,457,346


335762

## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel. Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal. The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

Selection here is based on **training trade_pnl** (not copyability).


In [6]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS, token_df): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

5590153 trades in 0.parquet
3797708 trades after merging with token_df for 0.parquet
5248322 trades in 1.parquet
3880988 trades after merging with token_df for 1.parquet
5315594 trades in 2.parquet
4329663 trades after merging with token_df for 2.parquet
5747407 trades in 3.parquet
4487775 trades after merging with token_df for 3.parquet
5673390 trades in 5.parquet
6105679 trades in 4.parquet
4157566 trades after merging with token_df for 5.parquet
4726099 trades after merging with token_df for 4.parquet
6045247 trades in 6.parquet
5812593 trades in 7.parquet
5490674 trades in 8.parquet
4863569 trades after merging with token_df for 6.parquet
5331258 trades in 9.parquet
4283044 trades after merging with token_df for 8.parquet
4601498 trades after merging with token_df for 7.parquet
3947259 trades after merging with token_df for 9.parquet
Enriched 0.parquet with copyable_qty
Enriched 1.parquet with copyable_qty
6562927 trades in a.parquet
4797551 trades after merging with token_df for a

In [7]:
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 200)

# MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

# df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
# df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [8]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [9]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [10]:
from polymarket_analysis.preprocessing.shard_processor import select_top_wallets_shard

# ── load token resolution DataFrame ─────────────────────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard (trade_pnl)...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            select_top_wallets_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_pct=0.04,
            selection_pnl="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")


Phase 1 — selecting top-4% wallets per shard (trade_pnl)...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Shard top 4.00% threshold: 87.22 USDC
Processing shard enriched_2.parquet...
Shard top 4.00% threshold: 86.20 USDC
Processing shard enriched_3.parquet...
Processing shard enriched_4.parquet...
Shard top 4.00% threshold: 87.00 USDC
Processing shard enriched_5.parquet...
Shard top 4.00% threshold: 125.90 USDC
  4/16 shards | raw: 16,496,134 | in-range: 16,496,134 | wallet union so far: 24,932
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 108.57 USDC
Processing shard enriched_7.parquet...
Shard top 4.00% threshold: 69.28 USDC
Processing shard enriched_8.parquet...
Shard top 4.00% threshold: 111.73 USDC
Shard top 4.00% threshold: 90.02 USDC
Processing shard enriched_9.parquet...
  8/16 shards | raw: 34,844,866 | in-range: 34,844,866 | wallet union so far: 40,898
Processing shard enriched_a.parquet...
Shard top 4.00% threshold: 98.00 U

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [11]:
from polymarket_analysis.preprocessing.shard_processor import enrich_and_group_shard

# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training trade_pnl per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            enrich_and_group_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_wallets,
            wallet_pnl_metric="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty     = ("copyable_qty",     "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)


Phase 2 — grouping and filtering shards...


  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 26,069,555
  Unique wallets in grouped:                   61,146


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x02e401502bdd6ef717b5c1bc577898ee7b2178636363...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-19 19:11:30+00:00,0x843fc1f23d8cfd3df343031312a8a3f0a930ad118c0c...,Yes,False,0.00,625.00,625.00,50.00,50.00,0.00,1,-50.00,-41.98,524.77,488.04,3.00,0.08
1,0x8d8589c941cedad7f6b153977a41da6d014c9fa24aec...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-21 11:37:40+00:00,0x23c805587933d498ebf46638ad13a1ad3a4f769a2cab...,Yes,False,0.00,51.85,51.85,20.00,20.00,0.00,3,-20.00,-0.00,0.00,0.00,0.00,0.39
2,0x38e32d9e7ccc04d6a6460d0ae282ec09620086e975aa...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,SELL,2026-01-21 11:48:20+00:00,0x23c805587933d498ebf46638ad13a1ad3a4f769a2cab...,Yes,False,0.00,43.28,51.84,18.85,18.85,0.00,5,18.85,0.00,0.00,0.00,0.00,0.36
3,0x73d67286b2307e92fbf1584420929eaaa0ad126872df...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-21 14:51:12+00:00,0x400ddb4652a3aef5f03c8aa42a530a940caaf5b6fa5b...,Yes,False,0.00,32.82,32.82,12.14,12.14,0.00,1,-12.14,0.00,-0.00,-0.00,0.00,0.37
4,0xf40ce36891d47b018768f37ad4373997960b859686d5...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-22 09:17:00+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Yes,True,1.00,300.00,300.00,51.00,51.00,300.00,1,249.00,100.68,121.30,19.41,2.00,0.17


## 4 · Phase 3: apply final trade-PnL cut-off and export

Compute the true cross-shard training P&L per wallet. Use the **median** of the
per-shard trade-PnL thresholds from Phase 1 as the export cut-off.


In [12]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",       "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        trade_pnl       = ("trade_pnl",       "sum"),
        copyable_pnl    = ("copyable_pnl",    "sum"),
    )
    .sort_values("trade_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard trade_pnl thresholds from Phase 1 ───
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(
    f"wallet_summary['trade_pnl'] quantiles:\n"
    f"{wallet_summary['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

final_wallets = set(
    wallet_summary[
        wallet_summary["trade_pnl"] >= pnl_cutoff
    ]["wallet"]
)

print(f"\nFinal selected wallets (trade-PnL filter): {len(final_wallets):,}")
print(
    f"final_wallets['trade_pnl'] quantiles:\n"
    f"{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard trade-PnL median cut-off:    {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing trade-PnL cut-off:     {len(final_wallets):,}")
wallet_summary.head(5)



Unique wallets in training data: 61,146
wallet_summary['trade_pnl'] quantiles:
0.01   -19266.20
0.10    -1256.09
0.20     -302.86
0.30      -17.97
0.40       96.48
0.50      153.57
0.60      240.00
0.70      406.47
0.80      761.28
0.90     2117.44
0.99    37041.74
Name: trade_pnl, dtype: float64

Final selected wallets (trade-PnL filter): 37,411
final_wallets['trade_pnl'] quantiles:
0.01      93.36
0.10     123.45
0.20     161.29
0.30     210.80
0.40     282.79
0.50     393.37
0.60     552.33
0.70     875.76
0.80    1574.10
0.90    4278.69
0.99   57278.81
Name: trade_pnl, dtype: float64
Unique wallets after Phase 2 union:    61,146
Per-shard trade-PnL median cut-off:    89.41 USDC
Wallets passing trade-PnL cut-off:     37,411


,wallet,num_markets,num_trades,total_cost_usdc,trade_pnl,copyable_pnl
0,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,605,51697,18229909.25,1439468.81,443561.76
1,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,42,2385,3878059.88,1304494.87,501531.92
2,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,223,8677,34701349.80,1168911.13,318346.69
3,0xbacd00c9080a82ded56f504ee8810af732b0ab35,1384,162193,35052547.01,1163773.88,8604.58
4,0x0a854897a06d4999e5b2dde5693609f1428ffe9d,50,5374,3443574.09,1082906.59,148610.70


## 5 · Market-level volume summary

In [13]:
# join market metadata (question, end_date) for display
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)
grouped_meta = grouped.merge(
    markets_meta,
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Processing 74 market file(s)…
Saved 1,322,218 markets → /Users/vobornij/projects/polymarket/data/markets_processed/markets.parquet
Unique markets: 74,325


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69...,US forces enter Iran by April 30?,2026-04-30T00:00:00Z,5266,186167,224623327.64,223643666.46
1,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,US x Iran ceasefire by April 7?,2026-04-07T00:00:00Z,4398,146739,146992346.04,152533955.28
2,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3...,US government shutdown Saturday?,2026-01-31T00:00:00Z,5101,248682,134670099.38,138187929.98
3,0x655e5ca101c466b6293aa15e06173b78b293221803d5...,Will Zelenskyy wear a suit before July?,2025-06-30T00:00:00Z,2303,72406,130903948.69,131699125.23
4,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,"US x Iran ceasefire extended by April 22, 2026?",2026-04-21T00:00:00Z,1700,52618,119435760.70,119143176.58
5,0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad1...,"Russia x Ukraine ceasefire by May 31, 2026?",2026-05-31T00:00:00Z,1175,32929,107942701.70,107322255.83
6,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Khamenei out as Supreme Leader of Iran by Febr...,2026-02-28T00:00:00Z,6024,195814,105555617.24,115376506.89
7,0x7cb525e831729325d651017f81cbcb6f1adde5011c7b...,Netanyahu out by March 31?,2026-03-31T00:00:00Z,2721,109424,94923136.11,95937007.13
8,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,"US strikes Iran by February 28, 2026?",2026-01-31T00:00:00Z,7423,198837,73181868.02,89799119.12
9,0xef8cf8b45ef7e3a607a72b6e1d56bede869fdd81795b...,TikTok banned in the US before May 2025?,2025-04-30T00:00:00Z,1422,41296,72669608.78,73377558.41


## 6 · Wallet statistics (quantiles)

In [14]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,61,1.00,1.00,-65204.31,-124073.69,61085
0.01,611,1.00,1.00,-11478.44,-19266.20,60535
0.05,3057,2.00,1.00,-2195.83,-3502.48,58089
0.10,6115,4.00,1.00,-849.74,-1256.09,55031
0.25,15286,13.00,3.00,-123.26,-126.03,45860
0.50,30573,49.00,8.00,0.03,153.57,30573
0.75,45860,169.00,23.00,188.33,535.47,15286
0.90,55031,550.00,66.00,777.90,2117.44,6115
0.95,58089,1241.80,124.00,1885.86,5664.93,3057


## 7 · Full enriched trade table

In [15]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x02e401502bdd6ef717b5c1bc577898ee7b2178636363...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-19 19:11:30+00:00,0x843fc1f23d8cfd3df343031312a8a3f0a930ad118c0c...,Yes,625.00,625.00,0.08,False,0.00,50.00,0.00,-50.00,-41.98,1
1,0x8d8589c941cedad7f6b153977a41da6d014c9fa24aec...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-21 11:37:40+00:00,0x23c805587933d498ebf46638ad13a1ad3a4f769a2cab...,Yes,51.85,51.85,0.39,False,0.00,20.00,0.00,-20.00,-0.00,3
2,0x38e32d9e7ccc04d6a6460d0ae282ec09620086e975aa...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,SELL,2026-01-21 11:48:20+00:00,0x23c805587933d498ebf46638ad13a1ad3a4f769a2cab...,Yes,43.28,51.84,0.36,False,0.00,18.85,0.00,18.85,0.00,5
3,0x73d67286b2307e92fbf1584420929eaaa0ad126872df...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-21 14:51:12+00:00,0x400ddb4652a3aef5f03c8aa42a530a940caaf5b6fa5b...,Yes,32.82,32.82,0.37,False,0.00,12.14,0.00,-12.14,0.00,1
4,0xf40ce36891d47b018768f37ad4373997960b859686d5...,0x0003ca93b9c5989bab65dd3084b35fbcec965260,BUY,2026-01-22 09:17:00+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Yes,300.00,300.00,0.17,True,1.00,51.00,300.00,249.00,100.68,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26069550,0x3bcd973a3ee5617ac8b063536a42ebc524c9863ceaab...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-08 18:51:25+00:00,0xc734f61dbb7fbf5ff7afac8c18b1f05f13c4b9b408f8...,Yes,2500.00,1000.00,0.01,False,0.00,6.00,0.00,-6.00,-6.00,1
26069551,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c799...,Yes,200.00,200.00,0.02,False,0.00,4.00,0.00,-4.00,-4.00,1
26069552,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c799...,Yes,4200.00,4000.00,0.00,False,0.00,12.00,0.00,-12.00,-12.00,1
26069553,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Yes,530.00,50.00,0.51,True,1.00,25.35,50.00,-24.65,-24.65,2


## 8 · Export: apply trade-PnL cut-off and write partitioned parquet

Filter grouped trades to `final_wallets` (wallets above the median per-shard
trade-PnL threshold), then write one parquet shard per first hex character of
`condition_id` after `0x`.


In [16]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 37,411


In [17]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  17,649,785
  training rows: 13,744,622
  test rows:     3,905,163
Output shards:  16
  0.parquet  (941,606 rows)
  1.parquet  (985,240 rows)
  2.parquet  (1,108,503 rows)
  3.parquet  (1,148,148 rows)
  4.parquet  (1,251,077 rows)
  5.parquet  (1,046,522 rows)
  6.parquet  (1,310,887 rows)
  7.parquet  (1,191,348 rows)
  8.parquet  (1,117,651 rows)
  9.parquet  (990,902 rows)
  a.parquet  (1,209,823 rows)
  b.parquet  (1,125,880 rows)
  c.parquet  (1,085,678 rows)
  d.parquet  (1,125,400 rows)
  e.parquet  (951,903 rows)
  f.parquet  (1,059,217 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [18]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [19]:
df = pd.read_parquet(ENRICHED_TRADES_DIR/'enriched_3.parquet')
len(df)

4487775

In [20]:
len(df[(df['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')])

268526

In [21]:
dfs = []
for f in sorted(ENRICHED_TRADES_DIR.glob("*.parquet")):
    dfp = pd.read_parquet(f)
    dfs.append(dfp[dfp['wallet'] == '0x96489abcb9f583d6835c8ef95ffc923d05a86825'])

df_full = pd.concat(dfs, ignore_index=True)

In [22]:
pd.set_option('display.max_columns', None)

In [23]:
enriched = df_full.copy()
enriched["dt"] = pd.to_datetime(enriched["block_timestamp"], unit="s", utc=True)
enriched['final_price'] = enriched['token_winner'] * 1
enriched["final_value_usdc"] = enriched["quantity"] * enriched["final_price"]
enriched["price_x_qty"] = enriched["price"] * enriched["quantity"]

_GROUP_KEYS = ["tx_hash", "wallet", "side"]
# Group fills → one row per tx_hash × wallet × side
grouped = (
    enriched.groupby(_GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",              "first"),
        condition_id     = ("condition_id",    "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("quantity",         "sum"),
        price_x_qty_sum  = ("price_x_qty",     "sum"),
        trade_value_usdc = ("usdc_amount",      "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("log_index",        "count"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]

is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = np.where(
    is_buy,
    grouped["final_value_usdc"] - grouped["trade_value_usdc"],
    grouped["trade_value_usdc"] - grouped["final_value_usdc"],
)

In [24]:
grouped.groupby('condition_id').agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum'),
    total_pnl    = ('trade_pnl', 'sum'),
).sort_values('total_pnl', ascending=True).head(10)

,total_trades,total_volume,total_pnl
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,2838,1589594.90,-1589594.90
0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,1443,745147.28,-631003.04
0x4b02efe53e631ada84681303fd66d79ad615f3d2b6a28b4633d43d935f89af58,1555,585240.20,-577608.94
0xdc4b9ecbfac2c8e98fe18b35c4578a3c64e34ddfe16b1cf8a98ab3ad9d06c088,962,463369.04,-463369.04
0x15aa3c1259a716915e068a0d63c3885d2301d29e8982cbb1717ecb9b63d02d95,857,448182.82,-448182.82
0x70909f0ba8256a89c301da58812ae47203df54957a07c7f8b10235e877ad63c2,1256,477185.87,-419092.55
0x291f0f6023efa933d36cc80fd55f9d176309a92b61b9567fd4f044a8b21873fb,769,409423.70,-409423.70
0x1db02ba50e2312a62b4104de691cc7a76065d8d0da40decf93eb1b914a3217b7,726,577558.55,-347391.11
0x797d586ad45522306490b0cc9b2f21bdf957f3843476fae99f3bcc2cec83b74b,2524,336785.66,-334649.02


In [25]:
grouped.groupby(grouped['dt'].dt.date).agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum')
).sort_index(ascending=False).head(10)

,total_trades,total_volume
dt,,
2026-05-30,10,475.93
2026-05-29,56,1681.89
2026-05-28,57,1541.66
2026-05-27,108,6537.51
2026-05-26,208,37017.44
2026-05-25,71,22381.52
2026-05-24,71,56545.01
2026-05-23,86,27011.16
2026-05-22,136,50160.66


In [26]:
df = pd.read_parquet('/Users/vobornij/projects/polymarket/data/polygon/2026-05-04.parquet')

In [27]:
df.tail()

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,protocol_version,side,token_id,maker_amount_filled,taker_amount_filled,fee,builder,metadata
5989041,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5281,1777939198,0x7cd92feca2c9c2aa649cd0c0b34644b70f0ceba49ad9b6b9bd3c33f4685ad70c,0x106a48725a926362bbb137d4c8bccedbaa6aacee,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,10000900,10990000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989042,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5283,1777939198,0xf6d86802a56c0d0b6595bcb328576bb958732775eb5ceeac3aad1e6ec44a122f,0x8da42eb2e2c1eba87794e4c30bc689513b90f3d1,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,3649100,4010000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989043,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5287,1777939198,0xce23af9a8098a0eb0e69f8ddea7fdd4921d52dd73ce6b3c59c5204a885a7c137,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0xe111180000d2663c0091e4f400237545b87b996b,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,4752900,52810000,311410,0x4898df15ec6590495dc6c0fedf951ade3e64001d47f9caf44a64e86fc11959df,0x0000000000000000000000000000000000000000000000000000000000000000
5989044,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5303,1777939198,0x81f68266bb8d09be6ffce910ea780b506d9ef2540bf34a965f2aaf048e9e38b7,0xae3db1cc155e1b2a5420e256c42cd1649b5a6b95,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,14338260,179228250,0,0x8ba7e2a23e33175c53fccfa30261a35734772d5909e4e4f484f204bfde446c7a,0x0000000000000000000000000000000000000000000000000000000000000000
5989045,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5307,1777939198,0x012d6a71961ef6008f0bb1e8646c7b74b4a0eca2bf7ebb6358541c0d8f01287c,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0xe111180000d2663c0091e4f400237545b87b996b,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,164889990,179228250,949760,0x9fedc0b0702ca6cb294e5321d9491b1e38b8bd2b463a7f7b06df8e6d7553cd18,0x0000000000000000000000000000000000000000000000000000000000000000
